In [34]:
import os
import sys
import time
import logging
import datetime
import glob
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from io import StringIO
from Bio import Entrez
from Bio.Seq import Seq
import requests
import time
import random
import subprocess

#### Helper functions

In [ ]:
# run external command
def run(cmd, outfile=None, check=True):
    """run external command"""
    cmd = [str(c) for c in cmd]
    log.info("$ " + " ".join(cmd) + (f"  > {outfile}" if outfile else ""))
    if outfile:
        with open(outfile, "w") as fh:
            res = subprocess.run(cmd, stdout=fh, stderr=subprocess.PIPE, text=True)
    else:
        res = subprocess.run(cmd, capture_output=True, text=True)
    err = (res.stderr or "").strip()
    if err:
        log.info(err[:1500])
    if res.returncode != 0:
        log.error(f"failed rc = {res.returncode}: {''.join(cmd)}")
        if check:
            raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return res

In [ ]:
# logging
LOG_PATH = WD / "logs" / "v2r-phmm.log"
logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(sys.stdout)
    ],
    force = True,
)
log = logging.getLogger("v2r-phmm")
log.info(f"---v2r phmm seed collection | {datetime.datetime.now().isoformat()}---")
log.info(f"working directory : {WD}")


2026-05-22 17:00:36,263 | INFO | ---v2r phmm seed collection | 2026-05-22T17:00:36.263447---
2026-05-22 17:00:36,264 | INFO | working directory : /hpcfs/users/a1864358/sanders_lab/v2r_hmm


## Seed curation

In [ ]:
Entrez.email = "13billy.trim13@gmail.com"
"""dont hardcode"""
# Entrez.api_key = os.environ["NCBI_API_KEY"]  
SLEEP = 0.34

WD = Path("/hpcfs/users/a1864358/sanders_lab/v2r_hmm")

In [ ]:
def ncbi_fetch(query, db="protein", batch_size=500, source_tag=None):
    # fetch ncbi protein sequences
    # returns list of SeqRecord
    handle = Entrez.esearch(db=db, term=query, usehistory="y")
    record = Entrez.read(handle)
    handle.close()
    time.sleep(SLEEP)
    count = int(record["Count"])
    webenv = record["WebEnv"]
    query_key = record["QueryKey"]
    log.info(f"found {count} records for {query}")
    
    if count == 0:
        log.warning(f"no records found for {query}")
        return []

    seqs = []
    for start in range(0, count, batch_size):
        time.sleep(SLEEP)
        handle = Entrez.efetch(
            db = db,
            rettype = "fasta",
            retmode = "text",
            retstart = start,
            retmax = batch_size,
            webenv = webenv,
            query_key = query_key,
        )
        batch = list(SeqIO.parse(handle, "fasta"))
        handle.close()
        seqs.extend(batch)
        log.info(f"fetched {len(batch)} sequences")
    
    if source_tag:
        for s in seqs:
            s.description = s.description.rstrip() + f"[source = {source_tag}]"
    
    log.info(f"-> {len(seqs)} sequences fetched for source={source_tag}")
    return seqs

In [ ]:
def fetch_uniprot(url, source_tag, len_min=700, len_max=1200, timeout=120):
    # Fetch fasta from UniProt rest stream, apply length filter, tag source
    log.info(f"  Fetching: {url}")
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()

    seqs = list(SeqIO.parse(StringIO(r.text), "fasta"))
    log.info(f"  Raw sequences: {len(seqs)}")

    seqs = [s for s in seqs if len_min <= len(s.seq) <= len_max]
    log.info(f"  After length filter ({len_min}–{len_max} aa): {len(seqs)}")

    for s in seqs:
        s.description = s.description.rstrip() + f" [source={source_tag}]"
    return seqs


def dedup_extend(target, seen, batch):
    added = 0
    for s in batch:
        if s.id not in seen:
            target.append(s)
            seen.add(s.id)
            added += 1
    return added


def ncbi_fetch_multi(queries, source_tag):
    seqs, seen = [], set()
    for label, q in queries:
        log.info(f"  subquery: {label}")
        batch = ncbi_fetch(q, source_tag=source_tag)
        added = dedup_extend(seqs, seen, batch)
        log.info(f"  added {added} unique sequences from subquery")
    return seqs


def merge_by_seqhash(*seq_lists):
    # dedup across NCBI/UniProt (different accessions, same protein)
    import hashlib

    seen, out = set(), []
    for seqs in seq_lists:
        for s in seqs:
            key = hashlib.md5(str(s.seq).upper().encode()).hexdigest()
            if key not in seen:
                seen.add(key)
                out.append(s)
    return out

In [ ]:
LEN_MIN, LEN_MAX = 700, 1200
SLEN = f"{LEN_MIN}:{LEN_MAX}[SLEN]"

In [ ]:
MOUSE_QUERIES = [
    (
        "Mouse Vmn2r* gene RefSeq",
        f'Vmn2r*[Gene Name] AND "Mus musculus"[Organism] AND RefSeq[Filter] AND {SLEN}',
    ),
    (
        "Mouse vomeronasal type-2 title",
        f'"vomeronasal type-2" AND "Mus musculus"[Organism] AND {SLEN}',
    ),
]
log.info("--- source 1: mouse Vmn2r =")
mouse_seqs = ncbi_fetch_multi(MOUSE_QUERIES, source_tag="mouse_vmn2r")
log.info(f"mouse total unique: {len(mouse_seqs)} sequences")

RAT_QUERIES = [
    (
        "Rat Vom2r* gene RefSeq",
        f'Vom2r*[Gene] AND "Rattus norvegicus"[Organism] AND RefSeq[Filter] AND {SLEN}',
    ),
    (
        "Rat vomeronasal type-2 title",
        f'"vomeronasal type-2" AND "Rattus norvegicus"[Organism] AND {SLEN}',
    ),
]
log.info("--- source 2: rat Vom2r =")
rat_seqs = ncbi_fetch_multi(RAT_QUERIES, source_tag="rat_vom2r")
log.info(f"rat total unique: {len(rat_seqs)} sequences")

UNIPROT_REVIEWED_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=protein_name%3A%22vomeronasal+type+2+receptor%22+AND+reviewed%3Atrue"
    "&format=fasta"
    "&compressed=false"
)
log.info("--- source 4: UniProt Swiss-Prot reviewed V2Rs =")
uniprot_reviewed = fetch_uniprot(
    UNIPROT_REVIEWED_URL,
    source_tag="uniprot_reviewed",
    len_min=LEN_MIN,
    len_max=LEN_MAX,
)
log.info(f"UniProt reviewed: {len(uniprot_reviewed)} sequences")

SQUAMATA_QUERIES = [
    (
        "Squamata Vmn2r* gene",
        f"txid8509[Organism] AND Vmn2r*[Gene Name] AND {SLEN}",
    ),
    (
        "Squamata vomeronasal type-2 title",
        f'txid8509[Organism] AND "vomeronasal type-2"[Title] AND {SLEN}',
    ),
]
log.info("--- source 5a: Squamata NCBI =")
squamata_ncbi = ncbi_fetch_multi(SQUAMATA_QUERIES, source_tag="squamata_ncbi")
log.info(f"squamata NCBI total unique: {len(squamata_ncbi)}")

UNIPROT_SQUAMATA_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=taxonomy_id%3A8509+AND+protein_name%3A%22vomeronasal+type+2+receptor%22"
    "&format=fasta"
    "&compressed=false"
)
UNIPROT_SQUAMATA_IPR_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=taxonomy_id%3A8509+AND+xref%3Ainterpro-IPR004073"
    "&format=fasta"
    "&compressed=false"
)
UNIPROT_SQUAMATA_PFAM_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=taxonomy_id%3A8509+AND+(xref%3Apfam-PF01094+AND+xref%3Apfam-PF00003)"
    "&format=fasta"
    "&compressed=false"
)
log.info("--- source 5b: Squamata UniProt =")
squamata_uniprot_name = fetch_uniprot(
    UNIPROT_SQUAMATA_URL,
    source_tag="squamata_uniprot_name",
    len_min=LEN_MIN,
    len_max=LEN_MAX,
)
squamata_uniprot_ipr = fetch_uniprot(
    UNIPROT_SQUAMATA_IPR_URL,
    source_tag="squamata_uniprot_ipr",
    len_min=LEN_MIN,
    len_max=LEN_MAX,
)
squamata_uniprot_pfam = fetch_uniprot(
    UNIPROT_SQUAMATA_PFAM_URL,
    source_tag="squamata_uniprot_pfam",
    len_min=LEN_MIN,
    len_max=LEN_MAX,
)
log.info(f"squamata UniProt (protein_name): {len(squamata_uniprot_name)}")
log.info(f"squamata UniProt (IPR004073): {len(squamata_uniprot_ipr)}")
log.info(f"squamata UniProt (dual Pfam): {len(squamata_uniprot_pfam)}")
log.info(f"squamata total unique : {sum(len(v) for v in (squamata_ncbi, squamata_uniprot_name, squamata_uniprot_ipr, squamata_uniprot_pfam))}")

UNIPROT_XENOPUS_URL = (
    "https://rest.uniprot.org/uniprotkb/stream"
    "?query=taxonomy_id%3A8355+AND+protein_name%3A%22vomeronasal+type+2+receptor%22"
    "&format=fasta"
    "&compressed=false"
)
log.info("--- source 7: Xenopus tropicalis outgroup =")
xenopus_seqs = fetch_uniprot(
    UNIPROT_XENOPUS_URL,
    source_tag="xenopus_v2r",
    len_min=LEN_MIN,
    len_max=LEN_MAX,
)
log.info(f"Xenopus outgroup: {len(xenopus_seqs)} sequences")

# summary
seed_sources = {
    "mouse_vmn2r": mouse_seqs,
    "rat_vom2r": rat_seqs,
    "uniprot_reviewed": uniprot_reviewed,
    "squamata_ncbi": squamata_ncbi,
    "squamata_uniprot_name": squamata_uniprot_name,
    "squamata_uniprot_ipr": squamata_uniprot_ipr,
    "squamata_uniprot_pfam": squamata_uniprot_pfam,
    "xenopus_v2r": xenopus_seqs,
}
for name, seqs in seed_sources.items():
    log.info(f"  {name}: {len(seqs)}")
log.info(f"Total raw (with cross-source duplicates): {sum(len(v) for v in seed_sources.values())}")


In [ ]:
# combine, dedup, write raw combined FASTA
    # dedup only by accession ID @ this stage

all_sources = {
    "mouse_vmn2r": mouse_seqs,
    "rat_vom2r": rat_seqs,
    "uniprot_reviewed": uniprot_reviewed,
    "squamata_ncbi": squamata_ncbi,
    "squamata_uniprot_name": squamata_uniprot_name,
    "squamata_uniprot_ipr": squamata_uniprot_ipr,
    "squamata_uniprot_pfam": squamata_uniprot_pfam,
    "xenopus_v2r": xenopus_seqs,
}

# merge + dedup by accession
combined = []
seen_ids = set()
for src, seqs in all_sources.items():
    added = 0
    for s in seqs:
        if s.id not in seen_ids:
            combined.append(s)
            seen_ids.add(s.id)
            added += 1
    log.info(f" after dedup: +{added} unique from {src}")
log.info(f"Total unique: {len(combined)}")

# write 
RAW_OUT = WD / "raw" / "01_seeds_combined_raw.fasta"
SeqIO.write(combined, RAW_OUT, "fasta")
log.info(f"Wrote raw combined fasta to {RAW_OUT} ({RAW_OUT.stat().st_size // 1024} KB)")

print(f" total sequence : {len(combined)}")
print(f" written to {RAW_OUT} ({RAW_OUT.stat().st_size // 1024} KB)")

2026-05-18 22:17:13,503 | INFO |  after dedup: +332 unique from mouse_vmn2r
2026-05-18 22:17:13,537 | INFO |  after dedup: +200 unique from rat_vom2r
2026-05-18 22:17:13,537 | INFO |  after dedup: +3 unique from uniprot_reviewed
2026-05-18 22:17:13,537 | INFO |  after dedup: +49 unique from squamata_ncbi
2026-05-18 22:17:13,538 | INFO |  after dedup: +689 unique from squamata_uniprot_name
2026-05-18 22:17:13,539 | INFO |  after dedup: +1295 unique from squamata_uniprot_ipr
2026-05-18 22:17:13,540 | INFO |  after dedup: +401 unique from squamata_uniprot_pfam
2026-05-18 22:17:13,540 | INFO |  after dedup: +75 unique from xenopus_v2r
2026-05-18 22:17:13,541 | INFO | Total unique: 3044
2026-05-18 22:17:13,560 | INFO | Wrote raw combined FASTA to /hpcfs/users/a1864358/sanders_lab/v2r_hmm/raw/01_seeds_combined_raw.fasta (2964 KB)
 total sequence : 3044
 written to /hpcfs/users/a1864358/sanders_lab/v2r_hmm/raw/01_seeds_combined_raw.fasta (2964 KB)


In [ ]:
!seqkit stats ./raw/01_seeds_combined_raw.fasta

file                               format  type     num_seqs    sum_len  min_len  avg_len  max_len
./raw/01_seeds_combined_raw.fasta  FASTA   Protein     3,044  2,551,807      700    838.3    1,200


### 'Decoy' genes

- to address cross-reactivity with closely related genes
- v2rs = class C GPCRs, which share unique + conserved structure -> decoys share sequence similarity
    - venus flytrap domain (VFD): extracellular domain that binds ligands
    - cysteine rich domain (CRD): connects VFD -> membrane
    - 7 transmembrane domain (7TM): spans cell membrane

**TAS1R_** = taste receptor type-1 ; have similar VFD
**CASR** = calcium sensing receptor ; Class C + VFD + 7TM
**GRM_** = metabotropic glutamate receptors; most abundant GPCR in vertebrates (neurotransmitters in brain) ; highly conserved
**GABBR_** = GABA-B receptors; inhibitory neurotransmitter receptors ; 7TM
**GPCR5/6_** = orphan class C GPCRs; similar structure to taste/calcium receptors

=> if candidate matches decoy almost as well as V2R -> rejected
    -> Gathering threshold = run the curated V2R HMM against decoy database -> identifies highest scoring decoy = best false positive -> `GA floor = max(decoy_bitscore) + buffer`

In [ ]:
"""
decoy panel
    > pull from uniprot swiss-prot reviewed
    > for:
        > seed filtering: drop candidate seed unless
              hmmscan_bitscore(seed) - hmmscan_bitscore(best_decoy) >= MARGIN_BITS
              (min separation from hard negatives; rejects ambiguous hits)
        > post-build threshold calibration: GA floor = max(decoy_bitscore) + buffer
              (gathering threshold must sit above the strongest decoy)
    targets:
        > TAS1R1:3
        > CASR
        > GRM1:8
        > GABBR1 & GABBR2
        > GPRC5A, GPRC5B, GPRC6A
"""

DECOY_GENES = [
    "TAS1R1",
    "TAS1R2",
    "TAS1R3",
    "CASR",
    "GRM1",
    "GRM2",
    "GRM3",
    "GRM4",
    "GRM5",
    "GRM6",
    "GRM7",
    "GRM8",
    "GABBR1",
    "GABBR2",
    "GPRC5A",
    "GPRC5B",
    "GPRC6A",    
]

DECOY_ORGS = [
    '"Mus musculus"[Organism]',
    '"Homo sapiens"[Organism]',
]

decoy_seqs = []
decoy_seen = set()

for gene in DECOY_GENES:
    for org in DECOY_ORGS:
        query = f"{gene} AND {org} AND RefSeq[Filter]"
        log.info(f"fetching decoy sequences for {gene} in {org}")
        batch = ncbi_fetch(query, source_tag="decoy")
        for s in batch:
            if s.id not in decoy_seen:  
                decoy_seqs.append(s)
                decoy_seen.add(s.id)

log.info(f"Total decoy sequences: {len(decoy_seqs)}")

# write decoy panel
decoy_out = WD / "raw" / "02_decoy_sequences.fasta"
SeqIO.write(decoy_seqs, decoy_out, "fasta")
log.info(f"Wrote decoy panel to {decoy_out} ({decoy_out.stat().st_size // 1024} KB)")

2026-05-18 22:17:13,893 | INFO | Fetching decoy sequences for TAS1R1 in "Mus musculus"[Organism]
2026-05-18 22:17:15,085 | INFO | found 5 records for TAS1R1 AND "Mus musculus"[Organism] AND RefSeq[Filter]
2026-05-18 22:17:16,330 | INFO | fetched 5 sequences
2026-05-18 22:17:16,330 | INFO | -> 5 sequences fetched for source=decoy
2026-05-18 22:17:16,331 | INFO | Fetching decoy sequences for TAS1R1 in "Homo sapiens"[Organism]
2026-05-18 22:17:17,393 | INFO | found 6 records for TAS1R1 AND "Homo sapiens"[Organism] AND RefSeq[Filter]
2026-05-18 22:17:18,866 | INFO | fetched 6 sequences
2026-05-18 22:17:18,867 | INFO | -> 6 sequences fetched for source=decoy
2026-05-18 22:17:18,867 | INFO | Fetching decoy sequences for TAS1R2 in "Mus musculus"[Organism]
2026-05-18 22:17:20,101 | INFO | found 2 records for TAS1R2 AND "Mus musculus"[Organism] AND RefSeq[Filter]
2026-05-18 22:17:21,009 | INFO | fetched 2 sequences
2026-05-18 22:17:21,009 | INFO | -> 2 sequences fetched for source=decoy
2026-05

In [ ]:
LOG="/hpcfs/users/a1864358/sanders_lab/v2r_hmm/logs/v2r-phmm.log"
!seqkit stats ./raw/02_decoy_sequences.fasta >> $LOG
!seqkit stats ./raw/01_seeds_combined_raw.fasta >> $LOG

- length filter >700aa
- remove duplicates
- cd hit @ 90%

In [ ]:
%%bash

WD="/hpcfs/users/a1864358/sanders_lab/v2r_hmm"
LOG="/hpcfs/users/a1864358/sanders_lab/v2r_hmm/logs/v2r-phmm.log"

# length filter
seqkit seq --min-len 700 --max-len 1600 \
    --remove-gaps \
    $WD/raw/01_seeds_combined_raw.fasta \
    > $WD/filtered/02_seeds_length_filtered.fasta

N=$(grep -c "^>" $WD/filtered/02_seeds_length_filtered.fasta)
seqkit stats -a $WD/filtered/02_seeds_length_filtered.fasta >> $LOG
M=$(grep -c "^>" $WD/raw/01_seeds_combined_raw.fasta)
echo "Before length filter (700-1600 aa): $M" >> $LOG
echo "After length filter (700-1600 aa): $N" >> $LOG
echo "$(date)" >> $LOG

# dedup
N_PRIOR=$(grep -c "^>" $WD/filtered/02_seeds_length_filtered.fasta)
seqkit rmdup -s \
    $WD/filtered/02_seeds_length_filtered.fasta \
    > $WD/filtered/03_seeds_deduped.fasta

N=$(grep -c "^>" $WD/filtered/03_seeds_deduped.fasta)
seqkit stats -a $WD/filtered/03_seeds_deduped.fasta >> $LOG
echo "After dedup: $N" >> $LOG
echo "$(date)" >> $LOG

# cd-hit
cd-hit \
    -i $WD/filtered/03_seeds_deduped.fasta \
    -o $WD/filtered/04_seeds_cdhit.fasta \
    -c 0.90 \
    -M 0 \
    -T 24

N=$(grep -c "^>" $WD/filtered/04_seeds_cdhit.fasta)
seqkit stats -a $WD/filtered/04_seeds_cdhit.fasta >> $LOG
echo "After cd-hit: $N" >> $LOG
echo "$(date)" >> $LOG  

seqkit stats -a $WD/filtered/04_seeds_cdhit.fasta >> $LOG
N=$(grep -c "^>" $WD/filtered/04_seeds_cdhit.fasta)
MIN=$(seqkit stats -T $WD/filtered/04_seeds_cdhit.fasta | awk 'NR==2{print $6}')
MAX=$(seqkit stats -T $WD/filtered/04_seeds_cdhit.fasta | awk 'NR==2{print $7}')
echo "N=$N, min=$MIN, max=$MAX" >> $LOG
echo "$(date)" >> $LOG

[INFO] 49 duplicated records removed
Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 22:00:32
Command: cd-hit -i
         /hpcfs/users/a1864358/sanders_lab/v2r_hmm/filtered/03_seeds_deduped.fasta
         -o
         /hpcfs/users/a1864358/sanders_lab/v2r_hmm/filtered/04_seeds_cdhit.fasta
         -c 0.90 -M 0 -T 24

Started: Fri May 22 16:58:00 2026
                            Output                              
----------------------------------------------------------------
total seq: 2995
longest and shortest : 1200 and 700
Total letters: 2509927
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 2M
Buffer          : 24 X 16M = 391M
Table           : 2 X 65M = 130M
Miscellaneous   : 0M
Total           : 524M

Table limit with the given memory limit:
Max number of representatives: 491581
Max number of word counting entries: 58326159

# comparing sequences from          0  to        115
---------- new table with       64 representatives
# comparing

## QC + filtering

- pseudogenes
- internal stop codons
- empty seq
- if >2% unknown aa

- strip trailing stop codons
- low quality seq
- remove V2R contaminants from decoy panel (by fasta descriptions)

In [ ]:
def n_seqs(fasta):
    # count seq
    return sum(1 for _ in SeqIO.parse(str(fasta), "fasta"))

"""
QC filter = remove pseudogenes (internal stop codons, empty seq, if >2% unknown (X))
"""
def clean_records(records, max_x_frac=0.02):
    # Drop sequences with internal stops or >2% X. Strip trailing stop.
    out = []
    for r in records:
        s = str(r.seq).rstrip("*")
        if "*" in s:
            continue
        if len(s) == 0 or s.count("X") / len(s) > max_x_frac:
            continue
        r.seq = Seq(s)
        out.append(r)
    return out

In [45]:
# clean + name-based filter

CAND_IN      = WD / "filtered" / "04_seeds_cdhit.fasta"
CAND_CLEAN   = WD / "filtered" / "05_seeds_clean.fasta"
DECOY_CLEAN  = WD / "filtered" / "05_decoy_clean.fasta"

# clean candidates
cand_recs = list(SeqIO.parse(CAND_IN, "fasta"))
cand_clean = clean_records(cand_recs)
log.info(f"Stage A: {len(cand_recs)} clean candidates → {len(cand_clean)} kept ({len(cand_recs) - len(cand_clean)} dropped by name)")

# filter out V2R contaminants from decoy panel
decoy_recs = list(SeqIO.parse(decoy_out, "fasta"))
decoy_clean_filtered = clean_records(decoy_recs)

# remove records with "vomeronasal" or "Vmn2r" or "Vom2r" in description
decoy_final = [
    r for r in decoy_clean_filtered 
    if not any(x in r.description.upper() for x in ["VOMERONASAL", "VMN2R", "VOM2R"])
]
log.info(f"Decoys: {len(decoy_recs)} clean records → {len(decoy_final)} kept ({len(decoy_clean_filtered) - len(decoy_final)} V2R contaminants dropped)")

# write
SeqIO.write(cand_clean, CAND_CLEAN, "fasta")
SeqIO.write(decoy_final, DECOY_CLEAN, "fasta")

len(decoy_final)

2026-05-22 17:35:30,479 | INFO | Stage A: 1974 clean candidates → 1888 kept (86 dropped by name)
2026-05-22 17:35:30,533 | INFO | Decoys: 388 clean records → 376 kept (12 V2R contaminants dropped)


376